# 05 — Property Value Processed Metrics

**Phase:** Processed Output Preparation  
**Notebook goal:** Prepare small, validated processed outputs from the City of Vancouver Property Tax Report that summarise assessed property value changes, for later comparison with residential permit activity.

---

## Purpose

This notebook computes four derived variables that measure year-over-year change in assessed property values:

- `current_total_assessed_value` — land + improvement assessed value for the current tax year
- `previous_total_assessed_value` — land + improvement assessed value for the prior tax year
- `absolute_value_change` — difference between current and previous totals
- `percentage_value_change` — that difference expressed as a percentage of the prior-year total

These variables were developed and validated individually in **Notebook 02**. This notebook consolidates that logic into a reusable function, validates it on a 1,000-row sample, and defines the planned processed output files.

> **Important caveat:** BC Assessment assessed values are not MLS sale prices, transaction prices, or market values. They are administrative valuations used as the basis for property tax calculations in British Columbia. Year-over-year changes in assessed value do not directly measure market appreciation or depreciation.

> **Scope:** This notebook does not yet process the full raw dataset. Sample logic is verified here first. Full-dataset processing will be added in a later step once the outputs and exports are confirmed correct on the sample.

## 1. Imports and Paths

We use `pathlib` for cross-platform path handling, `pandas` for all tabular operations, and `math` for the infinity check. All paths are derived from `PROJECT_ROOT` so the notebook works regardless of where it is launched from.

In [ ]:
from pathlib import Path
import pandas as pd
import math

# Project root is one level above the notebooks/ directory
PROJECT_ROOT       = Path(__file__).resolve().parent.parent if '__file__' in dir() else Path.cwd().parent
RAW_DATA_PATH      = PROJECT_ROOT / 'data' / 'raw' / 'property_tax_report_raw.csv'
PROCESSED_DATA_DIR = PROJECT_ROOT / 'data' / 'processed'

print(f'PROJECT_ROOT       : {PROJECT_ROOT}')
print(f'RAW_DATA_PATH      : {RAW_DATA_PATH}')
print(f'PROCESSED_DATA_DIR : {PROCESSED_DATA_DIR}')
print(f'Raw file exists    : {RAW_DATA_PATH.exists()}')
print(f'Processed dir exists: {PROCESSED_DATA_DIR.exists()}')

## 2. Constants

Source column names are confirmed in Notebook 02. Storing them as constants means a source rename requires changing only one line here, not every occurrence throughout the notebook.

In [ ]:
# Source column names — confirmed present in Notebook 02
COL_CURR_LAND        = 'CURRENT_LAND_VALUE'
COL_CURR_IMPROVEMENT = 'CURRENT_IMPROVEMENT_VALUE'
COL_PREV_LAND        = 'PREVIOUS_LAND_VALUE'
COL_PREV_IMPROVEMENT = 'PREVIOUS_IMPROVEMENT_VALUE'
COL_PID              = 'PID'

# Derived variable names
COL_CURR_TOTAL  = 'current_total_assessed_value'
COL_PREV_TOTAL  = 'previous_total_assessed_value'
COL_ABS_CHANGE  = 'absolute_value_change'
COL_PCT_CHANGE  = 'percentage_value_change'

# Sample size — full dataset not processed in this notebook
SAMPLE_ROWS = 1_000

print('Constants defined.')

## 3. Sample Load

The raw CSV file is approximately 443 MB, which makes loading the full dataset slow during development. We load only 1,000 rows here to verify column names and logic.

Key `read_csv` arguments:
- **`sep=';'`** — the file uses semicolons as delimiters, not commas.
- **`nrows=SAMPLE_ROWS`** — stops reading after 1,000 rows.
- **`encoding='utf-8-sig'`** — strips the byte-order mark (BOM) from the first column name.
- **`low_memory=False`** — reads the full sample before assigning `dtype`s, reducing mixed-type warnings.

In [ ]:
df = pd.read_csv(
    RAW_DATA_PATH,
    sep=';',
    nrows=SAMPLE_ROWS,
    encoding='utf-8-sig',
    low_memory=False,
)

print(f'Shape   : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Columns : {list(df.columns)}')
print()
display(df.head(5))

## 4. Column Verification

Before any feature engineering, we confirm that each required source column is present. These names were established in Notebook 02; we reuse them exactly.

In [ ]:
required_columns = [
    COL_PID,
    COL_CURR_LAND,
    COL_CURR_IMPROVEMENT,
    COL_PREV_LAND,
    COL_PREV_IMPROVEMENT,
]

all_present = True
for col in required_columns:
    status = 'FOUND  ' if col in df.columns else 'MISSING'
    print(f'  {status}: {col}')
    if 'MISSING' in status:
        all_present = False

print()
if all_present:
    print('All required columns present. Proceeding.')
else:
    print('WARNING: One or more required columns are missing. Do not proceed until resolved.')

## 5. Feature Engineering Function

The four derived variables are bundled into a single reusable function. This makes it straightforward to apply the same logic to the full dataset later without repeating code.

### Missing-value and safety rules

- **Totals** use `.sum(axis=1, min_count=2)`: both source components must be non-null for the total to be non-null. A partial total (one component present, one missing) is not computed — it would silently understate the full assessed value.
- **Absolute change** uses standard subtraction (`-`). Pandas propagates `NaN` through subtraction automatically, so any row missing either total produces a null change.
- **Percentage change** is computed only for rows where `previous_total_assessed_value` is strictly positive. Rows with a null, zero, or negative previous total receive a null percentage. This avoids `inf` from division by zero and sign-inverted results from a negative denominator.

In [ ]:
def compute_value_change_features(df: pd.DataFrame) -> pd.DataFrame:
    """Return a copy of df with four assessed-value change columns appended."""
    out = df.copy()

    # Current and previous assessed totals
    out[COL_CURR_TOTAL] = out[[COL_CURR_LAND, COL_CURR_IMPROVEMENT]].sum(axis=1, min_count=2)
    out[COL_PREV_TOTAL] = out[[COL_PREV_LAND, COL_PREV_IMPROVEMENT]].sum(axis=1, min_count=2)

    # Absolute year-over-year change; NaN propagates automatically
    out[COL_ABS_CHANGE] = out[COL_CURR_TOTAL] - out[COL_PREV_TOTAL]

    # Percentage change — only for rows with a strictly positive previous total
    out[COL_PCT_CHANGE] = float('nan')
    eligible = out[COL_PREV_TOTAL].notna() & (out[COL_PREV_TOTAL] > 0) & out[COL_ABS_CHANGE].notna()
    out.loc[eligible, COL_PCT_CHANGE] = (
        out.loc[eligible, COL_ABS_CHANGE] / out.loc[eligible, COL_PREV_TOTAL] * 100
    )

    return out


print('Function defined: compute_value_change_features')

### Apply the function to the sample

In [ ]:
df = compute_value_change_features(df)

derived_cols = [COL_CURR_TOTAL, COL_PREV_TOTAL, COL_ABS_CHANGE, COL_PCT_CHANGE]
for col in derived_cols:
    print(f'  {col}: dtype={df[col].dtype}')

print()
display(df[[COL_PID] + derived_cols].head(10))

## 6. Sample Validation Checks

Six validation checks confirm that the function produced correct outputs on the sample. No rows are removed or capped at this stage — outliers are documented, not excluded.

> **Why keep outliers?** Extreme value changes may reflect genuine events — redevelopment, subdivision, zoning reclassification — rather than data errors. Removing them before investigation could discard valid signal. They are flagged here for later review.

### Check 1 — Non-null and missing counts

In [ ]:
print(f'{"Column":<40} {"Non-null":>10} {"Missing":>10} {"Missing %":>12}')
print('-' * 76)
for col in derived_cols:
    non_null = df[col].notna().sum()
    missing  = df[col].isna().sum()
    pct      = missing / len(df) * 100
    print(f'{col:<40} {non_null:>10,} {missing:>10,} {pct:>11.1f}%')

### Check 2 — Negative value counts (for totals)

In [ ]:
for col in [COL_CURR_TOTAL, COL_PREV_TOTAL]:
    neg_count = (df[col] < 0).sum()
    status    = 'OK' if neg_count == 0 else f'WARNING: {neg_count} negative values'
    print(f'{col}: {status}')

### Check 3 — Distribution statistics for absolute and percentage changes

In [ ]:
print('absolute_value_change (CAD)')
print(f'  Min    : {df[COL_ABS_CHANGE].min():>18,.0f}')
print(f'  Median : {df[COL_ABS_CHANGE].median():>18,.0f}')
print(f'  Mean   : {df[COL_ABS_CHANGE].mean():>18,.0f}')
print(f'  Max    : {df[COL_ABS_CHANGE].max():>18,.0f}')
print()
print('percentage_value_change (%)')
print(f'  Min    : {df[COL_PCT_CHANGE].min():>18.4f}')
print(f'  Median : {df[COL_PCT_CHANGE].median():>18.4f}')
print(f'  Mean   : {df[COL_PCT_CHANGE].mean():>18.4f}')
print(f'  Max    : {df[COL_PCT_CHANGE].max():>18.4f}')

### Check 4 — Infinity values

`inf` and `-inf` are not caught by `.isna()` and must be checked separately. Their presence would indicate that a zero denominator slipped through the eligibility mask.

In [ ]:
for col in [COL_ABS_CHANGE, COL_PCT_CHANGE]:
    pos_inf = (df[col] == float('inf')).sum()
    neg_inf = (df[col] == float('-inf')).sum()
    total   = pos_inf + neg_inf
    status  = 'OK — no infinities' if total == 0 else f'WARNING: {total} infinity values'
    print(f'{col}: {status}')

### Check 5 — Extreme percentage changes (flagged for review, not removed)

Changes above +100% or below -50% are flagged. Common non-error causes: redevelopment, subdivision, land-use reclassification, or a very small prior-year denominator. These rows are documented here but not excluded from the dataset.

In [ ]:
EXTREME_HIGH_THRESHOLD = 100.0   # percent
EXTREME_LOW_THRESHOLD  = -50.0   # percent

extreme_high = (df[COL_PCT_CHANGE] > EXTREME_HIGH_THRESHOLD).sum()
extreme_low  = (df[COL_PCT_CHANGE] < EXTREME_LOW_THRESHOLD).sum()

print(f'Changes above +{EXTREME_HIGH_THRESHOLD:.0f}% : {extreme_high:,}')
print(f'Changes below {EXTREME_LOW_THRESHOLD:.0f}%  : {extreme_low:,}')
print()

review_cols = [COL_PID, COL_CURR_TOTAL, COL_PREV_TOTAL, COL_ABS_CHANGE, COL_PCT_CHANGE]

print('Five largest percentage changes (for review):')
display(df.loc[df[COL_PCT_CHANGE].nlargest(5).index, review_cols])

print()
print('Five smallest percentage changes (for review):')
display(df.loc[df[COL_PCT_CHANGE].nsmallest(5).index, review_cols])

### Check 6 — Arithmetic spot-check

For every row where both source totals are non-null, we independently re-compute the absolute change and percentage change using the raw formula and compare them to the derived columns. All values must match exactly.

In [ ]:
both_valid   = df[COL_CURR_TOTAL].notna() & df[COL_PREV_TOTAL].notna()
eligible_pct = both_valid & (df[COL_PREV_TOTAL] > 0)

# Absolute change
expected_abs   = df.loc[both_valid, COL_CURR_TOTAL] - df.loc[both_valid, COL_PREV_TOTAL]
abs_match      = (expected_abs == df.loc[both_valid, COL_ABS_CHANGE]).all()

# Percentage change
expected_pct   = (
    df.loc[eligible_pct, COL_ABS_CHANGE] / df.loc[eligible_pct, COL_PREV_TOTAL] * 100
)
pct_match      = (expected_pct == df.loc[eligible_pct, COL_PCT_CHANGE]).all()

print(f'Absolute change arithmetic check — rows checked: {both_valid.sum():,}, all match: {abs_match}')
print(f'Percentage change arithmetic check — rows checked: {eligible_pct.sum():,}, all match: {pct_match}')

## 7. Planned Processed Outputs

The cells below define the intended export paths and the logic that will produce each output. **No export is performed in this notebook.** Full-dataset processing and export will be added once the sample logic above is confirmed correct and the output schemas are approved.

---

### Planned output 1: `property_value_change_summary.csv`

**Target path:** `data/processed/property_value_change_summary.csv`  
**Expected rows:** One row per year (or one row total, depending on the aggregation confirmed later)  
**Description:** Aggregate statistics (count, median, mean, min, max) for `absolute_value_change` and `percentage_value_change` across the full dataset, grouped by assessment year if a year column is available.

```python
# PLANNED — do not execute yet
# summary = (
#     df_full
#     .groupby('TAX_LEVY_YEAR')[[COL_ABS_CHANGE, COL_PCT_CHANGE]]
#     .agg(['count', 'median', 'mean', 'min', 'max'])
#     .reset_index()
# )
# summary.to_csv(PROCESSED_DATA_DIR / 'property_value_change_summary.csv', index=False)
```

---

### Planned output 2: `property_value_change_distribution.csv`

**Target path:** `data/processed/property_value_change_distribution.csv`  
**Expected rows:** One row per percentile bucket or histogram bin  
**Description:** Distribution of `percentage_value_change` across the full dataset, capturing the spread of assessed value movements across Vancouver properties. Useful for visualising the histogram of changes.

```python
# PLANNED — do not execute yet
# bins = pd.cut(df_full[COL_PCT_CHANGE], bins=20)
# distribution = bins.value_counts().sort_index().reset_index()
# distribution.columns = ['pct_change_bin', 'count']
# distribution.to_csv(PROCESSED_DATA_DIR / 'property_value_change_distribution.csv', index=False)
```

## 8. Final Status

This cell prints a clear confirmation of what this notebook currently does and does not do.

In [ ]:
print('=' * 70)
print('NOTEBOOK STATUS')
print('=' * 70)
print()
print('This notebook currently validates sample logic ONLY.')
print(f'  Rows processed : {SAMPLE_ROWS:,} (sample — not the full raw dataset)')
print(f'  Raw file       : {RAW_DATA_PATH.name}')
print()
print('The full raw dataset has NOT been processed.')
print('No files have been written to data/processed/.')
print()
print('Next step: confirm sample output schemas, then apply')
print('compute_value_change_features() to the full dataset and export.')
print('=' * 70)